# 10 — End-to-end live deployment

`DeploymentPipeline` composes broker + feed + postback handler + strategy into a single start/stop lifecycle. This notebook runs it against fakes.

In [ ]:
import asyncio
from typing import Any
from ml4t.india.kite.client import KiteClient, AsyncKiteClient
from ml4t.india.kite.fake import FakeKiteClient
from ml4t.india.kite.rate_limit import KiteRateLimiter
from ml4t.india.live import KiteBroker, KiteTickerFeed, PostbackHandler
from ml4t.india.workflows import DeploymentPipeline

# Fakes: broker + ticker
sdk = FakeKiteClient()
broker = KiteBroker(AsyncKiteClient(KiteClient(sdk=sdk, rate_limiter=KiteRateLimiter())))

class FakeTicker:
    on_ticks = on_connect = on_close = on_error = None
    def __init__(self, *_): pass
    def connect(self, threaded=True): pass
    def close(self, *_, **__): pass
    def subscribe(self, tokens): pass
    def unsubscribe(self, tokens): pass
    def set_mode(self, m, t): pass
    def is_connected(self): return True

feed = KiteTickerFeed('k', 'a', ticker_factory=lambda *_: FakeTicker())
postbacks = PostbackHandler(api_secret='sekret', verify=False)

class MyStrategy:
    def on_tick(self, ticks: list[dict[str, Any]]) -> None:
        print('strategy got', len(ticks), 'ticks')
    def on_order(self, order: Any) -> None:
        print('strategy saw order', order.order_id, order.status)

pipeline = DeploymentPipeline(
    broker=broker, feed=feed,
    strategy=MyStrategy(),
    instrument_tokens=[738561, 2953217],
    postbacks=postbacks,
    subscription_mode='quote',
)

async def demo():
    await pipeline.start()
    print('pipeline started; subscribed to', pipeline.state.subscribed_tokens)
    await pipeline.stop()

asyncio.run(demo())
